## Save environment to project resource.

This notebook must be run from the project level, to store the environment under that project.

In [5]:
import pydicom
import os
import hashlib
from pyxnat import Interface

### Compute hash of the environment archive.

In [6]:
#Compute hash of the environment archive.
def compute_file_hash(path, length=16):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()[:length]

env_archive='/workspace/admin/micromamba/envs/duneaiM.tar.gz'
hash_value = compute_file_hash(env_archive, length=16)


In [7]:
hash_value

'9ed94cf51b14a51c'

### Check if environment resource exists in this project, and upload if not.

In [8]:
XNAT_HOST=os.getenv('XNAT_HOST')
PROJECT_ID=os.getenv('XNAT_ITEM_ID')
username=os.getenv('XNAT_USER')
password=os.getenv('XNAT_PASS')
print ('connecting to XNAT instance')
xnat=Interface(server=str(XNAT_HOST), user=username, password=password)
project=xnat.select.project(PROJECT_ID)

params = {
    "inbody": "true",
    "PROJECT_ID": PROJECT_ID,
    "extract": "false",
    "overwrite": "true"
}
env_basename = os.path.basename(env_archive)
remote_filename = f"{hash_value}_{env_basename}"
resource_label = 'ENVS'
resource = project.resource(resource_label)
remote_file = resource.file(remote_filename)

if not remote_file.exists():
    if not resource.exists():
        # create project resource "ENVS"
        resource.create()
    # upload file; file on server will be "<hash_value>_<env_archive_basename>"
    remote_file.put(env_archive, params=params)
    print(f"Uploaded {env_archive} as {remote_filename} to project {PROJECT_ID} resource {resource_label}")
else:
    print(f"File {remote_filename} already exists in project {PROJECT_ID} resource {resource_label}, not uploading.")

Uploaded /workspace/admin/micromamba/envs/duneaiM.tar.gz as 9ed94cf51b14a51c_duneaiM.tar.gz to project RIDER-LUNG-CT resource ENVS


In [9]:
#this cell holds a basic script to initialize container.
work_dir=/home/jovyan/work
mamba_dir=/home/jovyan/micromamba
#1. install micromamba.
curl -Ls https://micro.mamba.pm/install.sh | bash -s -- -b
mkdir -p $mamba_dir/envs
source ~/.bashrc
#2. install environment. Environment variables XNAT_HOST, XNAT_USER, XNAT_PASS, PROJECT, XNAT_ENV are initialized at the container command execution stage. 
curl -u "${XNAT_USER}:${XNAT_PASS}" -L \
  "${XNAT_HOST}/data/archive/projects/${PROJECT_ID}/resources/ENVS/files/${HASH_VALUE}_${ENV_BASENAME}" \
  -o "$mamba_dir/envs/${HASH_VALUE}_${ENV_BASENAME}"
mamba activate $ENV_BASENAME
#3. Read the base script 
